# Main Forecasting Model (cantine_presence)

Predicts daily cafeteria attendance 7 days in advance.

**Approach**: Predict `ratio = employees_count / office_present_pred` using predicted office
presence from the sub-model (Prompt 4), then convert: `count_pred = ratio_pred * office_present_pred`.

**Two architectures compared**:
- **A (Direct)**: calendar/menu/weather features only (no office presence features)
- **B (Cascade)**: includes predicted office presence as a feature

**Asymmetric metric** (under-prediction penalized 2x): `Score = mean(where(diff<0, -2*diff, diff))` where `diff = y_pred - y_true`.

**Inputs**: `features_train.csv`, `office_presence_train_oof.csv`, `office_presence_test_pred.csv`


In [1]:
import os, sys, warnings, json, pickle, time, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED = 42
HORIZON = 7
np.random.seed(SEED)

_cwd = Path().resolve()
REPO_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
PROC_DIR  = REPO_ROOT / 'data' / 'processed'
FIG_DIR   = REPO_ROOT / 'reports' / 'figures'
MODEL_DIR = REPO_ROOT / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


c:\Users\Drache Shop\anaconda3\envs\DM_ENV\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def asymmetric_cost(y_true, y_pred):
    diff = y_pred - y_true
    return np.mean(np.where(diff < 0, -2.0 * diff, diff))

def evaluate(y_true, y_pred, label=""):
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) == 0:
        print(f"  {label}: no valid pairs"); return {}
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    mape = np.mean(np.abs(yt - yp) / (np.abs(yt) + 1)) * 100
    r2 = 1 - np.sum((yt - yp)**2) / (np.sum((yt - np.mean(yt))**2) + 1e-9)
    asym = asymmetric_cost(yt, yp)
    print(f"  {label:30s}  RMSE={rmse:7.2f}  MAE={mae:7.2f}  MAPE={mape:5.1f}%  AsymCost={asym:7.4f}  R2={r2:.4f}")
    return dict(rmse=rmse, mae=mae, mape=mape, asym=asym, r2=r2)

print("Metric functions defined")

Metric functions defined


In [3]:
feat = pd.read_csv(PROC_DIR / 'features_train.csv', parse_dates=['Date'])
feat = feat.sort_values('Date').reset_index(drop=True)

oof_sub = pd.read_csv(PROC_DIR / 'office_presence_train_oof.csv', parse_dates=['Date'])
test_sub = pd.read_csv(PROC_DIR / 'office_presence_test_pred.csv', parse_dates=['Date'])

feat = feat.merge(oof_sub[['Date','office_presence_pred']].rename(columns={'office_presence_pred':'op_oof'}),
                  on='Date', how='left')
feat = feat.merge(test_sub[['Date','office_presence_pred']].rename(columns={'office_presence_pred':'op_test'}),
                  on='Date', how='left')
feat['office_present_pred'] = feat['op_oof'].fillna(feat['op_test']).fillna(feat['office_present'])
feat.drop(columns=['op_oof','op_test'], inplace=True)

feat['ratio_pred'] = feat['employees_count'] / feat['office_present_pred']
feat['ratio_pred_capped'] = feat['ratio_pred'].clip(0.30, 0.88)

EXCLUDE = {'Date','office_present','office_present_pred',
           'employees_count','ratio','ratio_capped','ratio_pred','ratio_pred_capped',
           'entrees','plat_principal_1','plat_principal_2','menu_combined',
           'weather_conditions','is_working_day','precipitation_type'}
ALL_FEAT_COLS = [c for c in feat.columns if c not in EXCLUDE and feat[c].dtype in ['int64','float64','int32','float32']]
NO_OP_FEATS   = [c for c in ALL_FEAT_COLS if not c.startswith('op_')]
WITH_OP_FEATS = ALL_FEAT_COLS[:]

print(f"Rows: {len(feat)}  |  {feat.Date.min().date()} to {feat.Date.max().date()}")
print(f"All features: {len(ALL_FEAT_COLS)}  |  No-office (Arch A): {len(NO_OP_FEATS)}  |  With-office (Arch B): {len(WITH_OP_FEATS)}")
print(f"Target range: [{feat.ratio_pred_capped.min():.3f}, {feat.ratio_pred_capped.max():.3f}]")

Rows: 604  |  2022-05-05 to 2024-12-31
All features: 126  |  No-office (Arch A): 102  |  With-office (Arch B): 126
Target range: [0.335, 0.880]


## Zero-Attendance Analysis


In [4]:
zero_mask = feat['employees_count'] == 0
print(f"Rows with employees_count == 0: {zero_mask.sum()}")

n = len(feat)
te_size = max(40, n // 8)
train_idx = np.arange(n - te_size - HORIZON)
test_idx  = np.arange(n - te_size, n)

TARGET = 'ratio_pred_capped'
FEAT_COLS_A = NO_OP_FEATS
FEAT_COLS_B = WITH_OP_FEATS

params_base = dict(objective='quantile', alpha=0.67, learning_rate=0.03,
                   num_leaves=31, min_child_samples=12, verbose=-1, seed=42)

Xa_tr = feat.iloc[train_idx][FEAT_COLS_A].values
y_tr  = feat.iloc[train_idx][TARGET].values
Xa_te = feat.iloc[test_idx][FEAT_COLS_A].values
y_te_c = feat.iloc[test_idx]['employees_count'].values
w_te   = feat.iloc[test_idx]['office_present_pred'].values

valid_tr = ~np.isnan(y_tr) & ~np.isnan(Xa_tr).any(axis=1)
m_all = lgb.train(params_base, lgb.Dataset(Xa_tr[valid_tr], label=y_tr[valid_tr]), num_boost_round=300)
pred_all = m_all.predict(Xa_te)

if zero_mask.sum() > 0:
    nz = ~zero_mask.values[train_idx] & valid_tr
    m_nz = lgb.train(params_base, lgb.Dataset(Xa_tr[nz], label=y_tr[nz]), num_boost_round=300)
    pred_nz = m_nz.predict(Xa_te)
    cost_all = asymmetric_cost(y_te_c, pred_all * w_te)
    cost_nz  = asymmetric_cost(y_te_c, pred_nz * w_te)
    print(f"With zeros:     AsymCost = {cost_all:.4f}")
    print(f"Without zeros:  AsymCost = {cost_nz:.4f}")
    EXCLUDE_ZEROS = cost_nz < cost_all
    print(f"-> {'Excluding zeros IMPROVES' if EXCLUDE_ZEROS else 'Including zeros is better or equal'}.")
else:
    EXCLUDE_ZEROS = False
    cost_nz = None
    print("No zero-attendance rows. Proceeding with all data.")

Rows with employees_count == 0: 0
No zero-attendance rows. Proceeding with all data.


## Architecture Comparison: Direct (A) vs Cascade (B)


In [5]:
tscv_comp = TimeSeriesSplit(n_splits=5, gap=HORIZON, test_size=max(40, n // 8))
costs_a, costs_b = [], []

for fold_i, (tr_i, te_i) in enumerate(tscv_comp.split(feat)):
    y_tr_f  = feat.iloc[tr_i][TARGET].values
    y_te_f  = feat.iloc[te_i][TARGET].values
    y_te_c  = feat.iloc[te_i]['employees_count'].values
    w_te_f  = feat.iloc[te_i]['office_present_pred'].values
    Xa_tr_f = feat.iloc[tr_i][NO_OP_FEATS].values
    Xa_te_f = feat.iloc[te_i][NO_OP_FEATS].values
    Xb_tr_f = feat.iloc[tr_i][WITH_OP_FEATS].values
    Xb_te_f = feat.iloc[te_i][WITH_OP_FEATS].values

    ok_tr = ~(np.isnan(y_tr_f) | np.isnan(Xa_tr_f).any(axis=1))
    ok_te = ~(np.isnan(y_te_f) | np.isnan(Xa_te_f).any(axis=1) | np.isnan(Xb_te_f).any(axis=1))
    if ok_tr.sum() < 20 or ok_te.sum() < 5:
        print(f"  Fold {fold_i}: skipped (too few valid rows)"); continue

    p = dict(objective='quantile', alpha=0.67, learning_rate=0.03, num_leaves=31,
             min_child_samples=12, verbose=-1, seed=42)
    m_a = lgb.train(p, lgb.Dataset(Xa_tr_f[ok_tr], label=y_tr_f[ok_tr]), num_boost_round=300)
    m_b = lgb.train(p, lgb.Dataset(Xb_tr_f[ok_tr], label=y_tr_f[ok_tr]), num_boost_round=300)

    p_a = m_a.predict(Xa_te_f[ok_te]) * w_te_f[ok_te]
    p_b = m_b.predict(Xb_te_f[ok_te]) * w_te_f[ok_te]
    costs_a.append(asymmetric_cost(y_te_c[ok_te], p_a))
    costs_b.append(asymmetric_cost(y_te_c[ok_te], p_b))
    print(f"  Fold {fold_i}: A={costs_a[-1]:.4f}  B={costs_b[-1]:.4f}  {'B wins' if costs_b[-1] < costs_a[-1] else 'A wins'}")

mean_a, mean_b = np.mean(costs_a), np.mean(costs_b)
USE_ARCH_B = mean_b < mean_a
FEAT_COLS = WITH_OP_FEATS if USE_ARCH_B else NO_OP_FEATS
ARCH_NAME = "B (Cascade)" if USE_ARCH_B else "A (Direct)"
print(f"\nArchitecture A mean: {mean_a:.4f}")
print(f"Architecture B mean: {mean_b:.4f}")
print(f"Selected: {ARCH_NAME}")

  Fold 0: skipped (too few valid rows)
  Fold 1: skipped (too few valid rows)
  Fold 2: skipped (too few valid rows)
  Fold 3: A=25.6100  B=23.5699  B wins
  Fold 4: A=31.6859  B=29.8879  B wins

Architecture A mean: 28.6479
Architecture B mean: 26.7289
Selected: B (Cascade)


## LightGBM Ensemble (24 sub-models: 6 alphas x 4 seeds)

Optuna tunes per fold (25 trials), then we train all alpha/seed combos.


In [6]:
tscv = TimeSeriesSplit(n_splits=5, gap=HORIZON, test_size=max(40, n // 8))
LGB_ALPHAS = [0.65, 0.68, 0.71, 0.74, 0.77, 0.80]
LGB_SEEDS  = [42, 7, 2024, 1337]
lgb_oof = np.full((n, len(LGB_ALPHAS)), np.nan)

fold_costs = []
all_best_params = []

for fold_i, (tr_i, te_i) in enumerate(tscv.split(feat)):
    y_tr_f = feat.iloc[tr_i][TARGET].values
    y_te_f = feat.iloc[te_i][TARGET].values
    y_te_c = feat.iloc[te_i]['employees_count'].values
    w_tr_f = feat.iloc[tr_i]['office_present_pred'].values
    w_te_f = feat.iloc[te_i]['office_present_pred'].values
    X_tr_f = feat.iloc[tr_i][FEAT_COLS].values
    X_te_f = feat.iloc[te_i][FEAT_COLS].values

    ok = ~(np.isnan(y_tr_f) | np.isnan(X_tr_f).any(axis=1))
    Xc, yc, wc = X_tr_f[ok], y_tr_f[ok], w_tr_f[ok]
    if len(Xc) < 30:
        print(f"Fold {fold_i}: skipped ({len(Xc)} clean rows < 30)"); continue

    print(f"\nFold {fold_i}: train={len(Xc)} test={len(te_i)}")

    n_tr_splt = max(20, int(len(Xc) * 0.85))
    Xtr, ytr, wtr = Xc[:n_tr_splt], yc[:n_tr_splt], wc[:n_tr_splt]
    Xval, yval, wval = Xc[n_tr_splt:], yc[n_tr_splt:], wc[n_tr_splt:]
    ec_clean = feat.iloc[tr_i]['employees_count'].values[ok]

    # Optuna
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))

    def objective(trial):
        p = dict(objective='quantile', alpha=0.71, metric='quantile',
                 learning_rate=trial.suggest_float('lr', 0.01, 0.06, log=True),
                 num_leaves=trial.suggest_int('nl', 25, 100),
                 max_depth=trial.suggest_int('md', 5, 8),
                 min_child_samples=trial.suggest_int('mcs', 8, 20),
                 subsample=trial.suggest_float('ss', 0.6, 1.0),
                 colsample_bytree=trial.suggest_float('cs', 0.6, 1.0),
                 reg_alpha=trial.suggest_float('ra', 1e-3, 10, log=True),
                 reg_lambda=trial.suggest_float('rl', 1e-3, 10, log=True),
                 verbose=-1, seed=42, n_jobs=-1)
        ds = lgb.Dataset(Xtr, label=ytr, weight=wtr)
        dv = lgb.Dataset(Xval, label=yval, weight=wval)
        m = lgb.train(p, ds, num_boost_round=1000, valid_sets=[dv],
                      callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(0)])
        return m.best_score.get('valid_0', {}).get('quantile', 1e9)

    study.optimize(objective, n_trials=25, timeout=700, catch=(Exception,))
    best = study.best_params if study.best_params else {'lr':0.03,'nl':31,'md':6,'mcs':12,'ss':0.8,'cs':0.8,'ra':0.1,'rl':1.0}
    all_best_params.append(best)
    print(f"  Optuna best: {study.best_value:.4f}  lr={best['lr']:.4f} nl={best['nl']}")

    # Multi-seed x alpha
    ok_te = ~(np.isnan(y_te_f) | np.isnan(X_te_f).any(axis=1))
    alphas_preds = []
    for ai, alpha in enumerate(LGB_ALPHAS):
        seed_preds = []
        for seed in LGB_SEEDS:
            p = dict(objective='quantile', alpha=alpha,
                     learning_rate=best['lr'], num_leaves=best['nl'],
                     max_depth=best['md'], min_child_samples=best['mcs'],
                     subsample=best['ss'], colsample_bytree=best['cs'],
                     reg_alpha=best['ra'], reg_lambda=best['rl'],
                     verbose=-1, seed=seed, n_jobs=-1)
            ds = lgb.Dataset(Xc, label=yc, weight=wc)
            if ok_te.sum() < 5:
                seed_preds.append(np.full(len(te_i), np.nan))
                continue
            m = lgb.train(p, ds, num_boost_round=1000)
            pred_f = np.full(len(te_i), np.nan)
            pred_f[ok_te] = m.predict(X_te_f[ok_te])
            seed_preds.append(pred_f)
        avg_pred = np.nanmean(seed_preds, axis=0)
        alphas_preds.append(avg_pred)
        lgb_oof[te_i, ai] = avg_pred

    ok_eval = ~np.isnan(alphas_preds[0])
    for ai, alpha in enumerate(LGB_ALPHAS):
        cost = asymmetric_cost(y_te_c[ok_eval], alphas_preds[ai][ok_eval] * w_te_f[ok_eval])
        print(f"  alpha={alpha:.2f}  AsymCost={cost:.4f}")

    ens = np.nanmean(alphas_preds, axis=0)
    ok_eval2 = ~np.isnan(ens)
    cost_ens = asymmetric_cost(y_te_c[ok_eval2], ens[ok_eval2] * w_te_f[ok_eval2])
    print(f"  LGB Ensemble AsymCost={cost_ens:.4f}")
    fold_costs.append(cost_ens)

print(f"\n{'='*60}")
print(f"LGB OOF mean AsymCost: {np.mean(fold_costs):.4f}")

[I 2026-08-27 09:33:48,857] A new study created in memory with name: no-name-2997b3ee-30c2-40cb-a8c7-fe4fb719ce4a


Fold 0: skipped (0 clean rows < 30)
Fold 1: skipped (0 clean rows < 30)
Fold 2: skipped (8 clean rows < 30)

Fold 3: train=83 test=75


[I 2026-08-27 09:33:49,152] Trial 0 finished with value: 0.0145383143324907 and parameters: {'lr': 0.01956360392823701, 'nl': 97, 'md': 7, 'mcs': 15, 'ss': 0.6624074561769746, 'cs': 0.662397808134481, 'ra': 0.0017073967431528124, 'rl': 2.9154431891537547}. Best is trial 0 with value: 0.0145383143324907.
[I 2026-08-27 09:33:49,218] Trial 1 finished with value: 0.016970878240482656 and parameters: {'lr': 0.0293601586511158, 'nl': 78, 'md': 5, 'mcs': 20, 'ss': 0.9329770563201687, 'cs': 0.6849356442713105, 'ra': 0.005337032762603957, 'rl': 0.00541524411940254}. Best is trial 0 with value: 0.0145383143324907.
[I 2026-08-27 09:33:49,475] Trial 2 finished with value: 0.014204667944030575 and parameters: {'lr': 0.017248307328108965, 'nl': 64, 'md': 6, 'mcs': 11, 'ss': 0.8447411578889518, 'cs': 0.6557975442608167, 'ra': 0.01474275315991467, 'rl': 0.029204338471814112}. Best is trial 2 with value: 0.014204667944030575.
[I 2026-08-27 09:33:49,761] Trial 3 finished with value: 0.014237655943262246

  Optuna best: 0.0124  lr=0.0215 nl=33


[I 2026-08-27 09:33:59,804] A new study created in memory with name: no-name-a8163c87-bf60-48c7-a137-632733315dd6
[I 2026-08-27 09:33:59,846] Trial 0 finished with value: 0.00971683379328183 and parameters: {'lr': 0.01956360392823701, 'nl': 97, 'md': 7, 'mcs': 15, 'ss': 0.6624074561769746, 'cs': 0.662397808134481, 'ra': 0.0017073967431528124, 'rl': 2.9154431891537547}. Best is trial 0 with value: 0.00971683379328183.
[I 2026-08-27 09:33:59,878] Trial 1 finished with value: 0.009686340445711929 and parameters: {'lr': 0.0293601586511158, 'nl': 78, 'md': 5, 'mcs': 20, 'ss': 0.9329770563201687, 'cs': 0.6849356442713105, 'ra': 0.005337032762603957, 'rl': 0.00541524411940254}. Best is trial 1 with value: 0.009686340445711929.
[I 2026-08-27 09:33:59,925] Trial 2 finished with value: 0.009636602335649525 and parameters: {'lr': 0.017248307328108965, 'nl': 64, 'md': 6, 'mcs': 11, 'ss': 0.8447411578889518, 'cs': 0.6557975442608167, 'ra': 0.01474275315991467, 'rl': 0.029204338471814112}. Best is t

  alpha=0.65  AsymCost=26.8990
  alpha=0.68  AsymCost=25.2194
  alpha=0.71  AsymCost=24.5071
  alpha=0.74  AsymCost=24.5070
  alpha=0.77  AsymCost=23.1463
  alpha=0.80  AsymCost=23.4804
  LGB Ensemble AsymCost=24.3808

Fold 4: train=158 test=75


[I 2026-08-27 09:34:00,003] Trial 4 finished with value: 0.009708874835098597 and parameters: {'lr': 0.011236213390203352, 'nl': 97, 'md': 8, 'mcs': 18, 'ss': 0.7218455076693483, 'cs': 0.6390688456025535, 'ra': 0.5456725485601477, 'rl': 0.057624872164786026}. Best is trial 2 with value: 0.009636602335649525.
[I 2026-08-27 09:34:00,050] Trial 5 finished with value: 0.009704478714314277 and parameters: {'lr': 0.012444120419299869, 'nl': 62, 'md': 5, 'mcs': 19, 'ss': 0.7035119926400067, 'cs': 0.8650089137415928, 'ra': 0.017654048052495083, 'rl': 0.12030178871154672}. Best is trial 2 with value: 0.009636602335649525.
[I 2026-08-27 09:34:00,087] Trial 6 finished with value: 0.009709869916901986 and parameters: {'lr': 0.026633196140794114, 'nl': 39, 'md': 8, 'mcs': 18, 'ss': 0.9757995766256756, 'cs': 0.9579309401710595, 'ra': 0.24637685958997463, 'rl': 4.869640941520899}. Best is trial 2 with value: 0.009636602335649525.
[I 2026-08-27 09:34:00,128] Trial 7 finished with value: 0.009693969003

  Optuna best: 0.0090  lr=0.0546 nl=51
  alpha=0.65  AsymCost=22.5106
  alpha=0.68  AsymCost=24.2195
  alpha=0.71  AsymCost=25.2357
  alpha=0.74  AsymCost=25.5587
  alpha=0.77  AsymCost=26.3201
  alpha=0.80  AsymCost=28.0283
  LGB Ensemble AsymCost=25.1495

LGB OOF mean AsymCost: 24.7651


In [7]:
oof_valid = ~np.isnan(lgb_oof[:, 0])
y_oof = feat.loc[oof_valid, 'employees_count'].values
w_oof = feat.loc[oof_valid, 'office_present_pred'].values

def cost_with_weights(w):
    pred = lgb_oof[oof_valid] @ w
    return asymmetric_cost(y_oof, pred * w_oof)

best_cost_a = 1e9
best_w_a = None
for _ in range(10):
    w0 = np.random.dirichlet(np.ones(len(LGB_ALPHAS)))
    res = minimize(cost_with_weights, w0, method='L-BFGS-B',
                   bounds=[(0, 1)] * len(LGB_ALPHAS),
                   options={'maxiter': 5000})
    w_res = res.x / (res.x.sum() + 1e-9)
    if res.fun < best_cost_a:
        best_cost_a = res.fun
        best_w_a = w_res

alpha_weights = dict(zip([f"a{a:.2f}" for a in LGB_ALPHAS], best_w_a))
print("Alpha weights:")
for k, v in alpha_weights.items():
    print(f"  {k}: {v:.4f}")
print(f"Optimized LGB OOF AsymCost: {best_cost_a:.4f}")

Alpha weights:
  a0.65: 0.1240
  a0.68: 0.1283
  a0.71: 0.1353
  a0.74: 0.0000
  a0.77: 0.5316
  a0.80: 0.0808
Optimized LGB OOF AsymCost: 24.3331


## XGBoost Ensemble (9 sub-models: 3 alphas x 3 seeds)


In [8]:
XGB_ALPHAS = [0.68, 0.71, 0.74]
XGB_SEEDS  = [42, 7, 2024]
xgb_oof = np.full((n, len(XGB_ALPHAS)), np.nan)

for fold_i, (tr_i, te_i) in enumerate(tscv.split(feat)):
    y_tr_f = feat.iloc[tr_i][TARGET].values
    y_te_f = feat.iloc[te_i][TARGET].values
    y_te_c = feat.iloc[te_i]['employees_count'].values
    w_tr_f = feat.iloc[tr_i]['office_present_pred'].values
    w_te_f = feat.iloc[te_i]['office_present_pred'].values
    X_tr_f = feat.iloc[tr_i][FEAT_COLS].values
    X_te_f = feat.iloc[te_i][FEAT_COLS].values

    ok = ~(np.isnan(y_tr_f) | np.isnan(X_tr_f).any(axis=1))
    Xc, yc, wc = X_tr_f[ok], y_tr_f[ok], w_tr_f[ok]
    if len(Xc) < 30: continue

    study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    ec_clean = feat.iloc[tr_i]['employees_count'].values[ok]

    def obj_xgb(trial):
        p = dict(objective='reg:quantileerror', quantile_alpha=0.71,
                 learning_rate=trial.suggest_float('lr', 0.01, 0.06, log=True),
                 max_depth=trial.suggest_int('md', 5, 8),
                 subsample=trial.suggest_float('ss', 0.6, 1.0),
                 colsample_bytree=trial.suggest_float('cs', 0.6, 1.0),
                 reg_alpha=trial.suggest_float('ra', 1e-3, 10, log=True),
                 reg_lambda=trial.suggest_float('rl', 1e-3, 10, log=True),
                 verbosity=0, seed=42, n_jobs=-1)
        ntr = max(20, int(len(Xc) * 0.85))
        m = xgb.XGBRegressor(**p, n_estimators=500)
        m.fit(Xc[:ntr], yc[:ntr], sample_weight=wc[:ntr], eval_set=[(Xc[ntr:], yc[ntr:])], verbose=False)
        pred = m.predict(Xc[ntr:])
        return asymmetric_cost(ec_clean[ntr:], pred * wc[ntr:])

    study_xgb.optimize(obj_xgb, n_trials=15, timeout=500, catch=(Exception,))
    try:
        best_xgb = study_xgb.best_params
    except ValueError:
        best_xgb = {'lr':0.03,'md':6,'ss':0.8,'cs':0.8,'ra':0.1,'rl':1.0}

    ok_te = ~(np.isnan(y_te_f) | np.isnan(X_te_f).any(axis=1))
    for ai, alpha in enumerate(XGB_ALPHAS):
        seed_preds = []
        for seed in XGB_SEEDS:
            p = dict(objective='reg:quantileerror', quantile_alpha=alpha,
                     learning_rate=best_xgb['lr'], max_depth=best_xgb['md'],
                     subsample=best_xgb['ss'], colsample_bytree=best_xgb['cs'],
                     reg_alpha=best_xgb['ra'], reg_lambda=best_xgb['rl'],
                     verbosity=0, seed=seed, n_jobs=-1)
            m = xgb.XGBRegressor(**p, n_estimators=500)
            m.fit(Xc, yc, sample_weight=wc, verbose=False)
            pred_f = np.full(len(te_i), np.nan)
            if ok_te.sum() > 0:
                pred_f[ok_te] = m.predict(X_te_f[ok_te])
            seed_preds.append(pred_f)
        xgb_oof[te_i, ai] = np.nanmean(seed_preds, axis=0)

    ok_eval = ~np.isnan(xgb_oof[te_i, 0])
    for ai, alpha in enumerate(XGB_ALPHAS):
        cost = asymmetric_cost(y_te_c[ok_eval], xgb_oof[te_i, ai][ok_eval] * w_te_f[ok_eval])
        print(f"  Fold {fold_i} XGB alpha={alpha:.2f}  AsymCost={cost:.4f}")

print("XGB OOF complete")

[I 2026-08-27 09:34:19,254] A new study created in memory with name: no-name-96e76397-4eb7-4b79-862a-50ed4dabbfa6
[I 2026-08-27 09:34:22,074] Trial 0 finished with value: 48.58535163677656 and parameters: {'lr': 0.01956360392823701, 'md': 8, 'ss': 0.892797576724562, 'cs': 0.8394633936788146, 'ra': 0.004207988669606638, 'rl': 0.004207053950287938}. Best is trial 0 with value: 48.58535163677656.
[I 2026-08-27 09:34:23,449] Trial 1 finished with value: 43.67539039483437 and parameters: {'lr': 0.011096801960099469, 'md': 8, 'ss': 0.8404460046972835, 'cs': 0.8832290311184181, 'ra': 0.0012087541473056963, 'rl': 7.579479953348009}. Best is trial 1 with value: 43.67539039483437.
[I 2026-08-27 09:34:24,469] Trial 2 finished with value: 40.34699484476676 and parameters: {'lr': 0.04443920508509773, 'md': 5, 'ss': 0.6727299868828402, 'cs': 0.6733618039413735, 'ra': 0.016480446427978974, 'rl': 0.12561043700013558}. Best is trial 2 with value: 40.34699484476676.
[I 2026-08-27 09:34:25,443] Trial 3 f

  Fold 3 XGB alpha=0.68  AsymCost=23.3435
  Fold 3 XGB alpha=0.71  AsymCost=23.3275
  Fold 3 XGB alpha=0.74  AsymCost=24.2411


[I 2026-08-27 09:34:44,533] Trial 0 finished with value: 15.793686737616857 and parameters: {'lr': 0.01956360392823701, 'md': 8, 'ss': 0.892797576724562, 'cs': 0.8394633936788146, 'ra': 0.004207988669606638, 'rl': 0.004207053950287938}. Best is trial 0 with value: 15.793686737616857.
[I 2026-08-27 09:34:46,125] Trial 1 finished with value: 17.65202673524618 and parameters: {'lr': 0.011096801960099469, 'md': 8, 'ss': 0.8404460046972835, 'cs': 0.8832290311184181, 'ra': 0.0012087541473056963, 'rl': 7.579479953348009}. Best is trial 0 with value: 15.793686737616857.
[I 2026-08-27 09:34:47,021] Trial 2 finished with value: 17.926960453391075 and parameters: {'lr': 0.04443920508509773, 'md': 5, 'ss': 0.6727299868828402, 'cs': 0.6733618039413735, 'ra': 0.016480446427978974, 'rl': 0.12561043700013558}. Best is trial 0 with value: 15.793686737616857.
[I 2026-08-27 09:34:48,070] Trial 3 finished with value: 18.16580628355344 and parameters: {'lr': 0.021682959386691958, 'md': 6, 'ss': 0.844741157

  Fold 4 XGB alpha=0.68  AsymCost=28.5284
  Fold 4 XGB alpha=0.71  AsymCost=28.2288
  Fold 4 XGB alpha=0.74  AsymCost=28.5990
XGB OOF complete


## CatBoost Ensemble (3 sub-models)


In [9]:
try:
    from catboost import CatBoostRegressor
    CB_SEEDS = [42, 7, 2024]
    cb_oof = np.full(n, np.nan)
    HAS_CATBOOST = True

    for fold_i, (tr_i, te_i) in enumerate(tscv.split(feat)):
        y_tr_f = feat.iloc[tr_i][TARGET].values
        y_te_f = feat.iloc[te_i][TARGET].values
        y_te_c = feat.iloc[te_i]['employees_count'].values
        w_tr_f = feat.iloc[tr_i]['office_present_pred'].values
        w_te_f = feat.iloc[te_i]['office_present_pred'].values
        X_tr_f = feat.iloc[tr_i][FEAT_COLS].values
        X_te_f = feat.iloc[te_i][FEAT_COLS].values

        ok = ~(np.isnan(y_tr_f) | np.isnan(X_tr_f).any(axis=1))
        Xc, yc, wc = X_tr_f[ok], y_tr_f[ok], w_tr_f[ok]
        if len(Xc) < 30: continue

        study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))

        def obj_cb(trial):
            p = dict(loss_function="Quantile:alpha=0.71",
                     learning_rate=trial.suggest_float('lr', 0.01, 0.1, log=True),
                     depth=trial.suggest_int('d', 4, 8),
                     l2_leaf_reg=trial.suggest_float('l2', 1, 10),
                     iterations=500, verbose=0, random_seed=42)
            ntr = max(20, int(len(Xc) * 0.85))
            m = CatBoostRegressor(**p)
            m.fit(Xc[:ntr], yc[:ntr], sample_weight=wc[:ntr], eval_set=(Xc[ntr:], yc[ntr:]), verbose=0)
            pred = m.predict(Xc[ntr:])
            return np.mean(np.abs(yc[ntr:] - pred))

        study_cb.optimize(obj_cb, n_trials=15, timeout=600, catch=(Exception,))
        try:
            best_cb = study_cb.best_params
        except ValueError:
            best_cb = {'lr':0.05,'d':6,'l2':3.0}

        seed_preds = []
        for seed in CB_SEEDS:
            m = CatBoostRegressor(loss_function="Quantile:alpha=0.71",
                                  learning_rate=best_cb['lr'], depth=best_cb['d'],
                                  l2_leaf_reg=best_cb['l2'], iterations=500, verbose=0, random_seed=seed)
            m.fit(Xc, yc, sample_weight=wc, verbose=0)
            pred_f = np.full(len(te_i), np.nan)
            ok_te = ~(np.isnan(y_te_f) | np.isnan(X_te_f).any(axis=1))
            if ok_te.sum() > 0:
                pred_f[ok_te] = m.predict(X_te_f[ok_te])
            seed_preds.append(pred_f)
        cb_oof[te_i] = np.nanmean(seed_preds, axis=0)
        ok_eval = ~np.isnan(cb_oof[te_i])
        cost = asymmetric_cost(y_te_c[ok_eval], cb_oof[te_i][ok_eval] * w_te_f[ok_eval])
        print(f"  Fold {fold_i} CatBoost AsymCost={cost:.4f}")

    print("CatBoost OOF complete")
except ImportError:
    HAS_CATBOOST = False
    cb_oof = None
    print("CatBoost not available — skipping")

[I 2026-08-27 09:35:14,360] A new study created in memory with name: no-name-f15697be-9a3c-4c10-9656-811bccdc6370
[I 2026-08-27 09:35:23,125] Trial 0 finished with value: 0.04706233303156863 and parameters: {'lr': 0.023688639503640783, 'd': 8, 'l2': 7.587945476302646}. Best is trial 0 with value: 0.04706233303156863.
[I 2026-08-27 09:35:24,852] Trial 1 finished with value: 0.053399887010631544 and parameters: {'lr': 0.03968793330444373, 'd': 4, 'l2': 2.403950683025824}. Best is trial 0 with value: 0.04706233303156863.
[I 2026-08-27 09:35:31,033] Trial 2 finished with value: 0.0495510609717079 and parameters: {'lr': 0.011430983876313222, 'd': 8, 'l2': 6.41003510568888}. Best is trial 0 with value: 0.04706233303156863.
[I 2026-08-27 09:35:32,769] Trial 3 finished with value: 0.05315692707689744 and parameters: {'lr': 0.051059032093947576, 'd': 4, 'l2': 9.72918866945795}. Best is trial 0 with value: 0.04706233303156863.
[I 2026-08-27 09:35:34,963] Trial 4 finished with value: 0.0523193470

  Fold 3 CatBoost AsymCost=23.0215


[I 2026-08-27 09:36:37,047] Trial 0 finished with value: 0.026146379850594984 and parameters: {'lr': 0.023688639503640783, 'd': 8, 'l2': 7.587945476302646}. Best is trial 0 with value: 0.026146379850594984.
[I 2026-08-27 09:36:40,193] Trial 1 finished with value: 0.025962819653470453 and parameters: {'lr': 0.03968793330444373, 'd': 4, 'l2': 2.403950683025824}. Best is trial 1 with value: 0.025962819653470453.
[I 2026-08-27 09:36:55,948] Trial 2 finished with value: 0.02639879318046304 and parameters: {'lr': 0.011430983876313222, 'd': 8, 'l2': 6.41003510568888}. Best is trial 1 with value: 0.025962819653470453.
[I 2026-08-27 09:36:58,798] Trial 3 finished with value: 0.027724625831594397 and parameters: {'lr': 0.051059032093947576, 'd': 4, 'l2': 9.72918866945795}. Best is trial 1 with value: 0.025962819653470453.
[I 2026-08-27 09:37:02,066] Trial 4 finished with value: 0.027381803073982112 and parameters: {'lr': 0.06798962421591129, 'd': 5, 'l2': 2.636424704863906}. Best is trial 1 with

  Fold 4 CatBoost AsymCost=25.7088
CatBoost OOF complete


## Three-Way Blend


In [10]:
lgb_weights_arr = np.array([alpha_weights[f"a{a:.2f}"] for a in LGB_ALPHAS])
lgb_ens_oof = np.nanmean(lgb_oof * lgb_weights_arr[None, :], axis=1)
xgb_ens_oof = np.nanmean(xgb_oof, axis=1)

if HAS_CATBOOST and cb_oof is not None:
    models_oof = np.column_stack([lgb_ens_oof, xgb_ens_oof, cb_oof])
    model_names = ['LGB', 'XGB', 'CatBoost']
else:
    models_oof = np.column_stack([lgb_ens_oof, xgb_ens_oof])
    model_names = ['LGB', 'XGB']

valid = ~np.isnan(models_oof).any(axis=1)
y_blend = feat.loc[valid, 'employees_count'].values
w_blend = feat.loc[valid, 'office_present_pred'].values

best_blend_cost = 1e9
best_blend_w = None
step = 0.02
nm = len(model_names)
for combo in itertools.product(range(0, int(1/step)+1), repeat=nm):
    w = np.array(combo, dtype=float) * step
    if w.sum() < 1e-9: continue
    w = w / w.sum()
    cost = asymmetric_cost(y_blend, (models_oof[valid] @ w) * w_blend)
    if cost < best_blend_cost:
        best_blend_cost = cost
        best_blend_w = w

print("Blend weights:")
for name, wi in zip(model_names, best_blend_w):
    print(f"  {name:10s}: {wi:.4f}")
print(f"Blended OOF AsymCost: {best_blend_cost:.4f}")

Blend weights:
  LGB       : 0.0159
  XGB       : 0.1905
  CatBoost  : 0.7937
Blended OOF AsymCost: 23.9795


## Two-Stage Calibration


In [11]:
blend_oof = models_oof @ best_blend_w
blend_count = blend_oof[valid] * w_blend
dow_oof = feat.loc[valid, 'dow'].values
resid_oof = y_blend - blend_count

# Stage 1: per-DOW additive offset (in count space)
offsets = {}
for d in [6, 0, 1, 2, 3]:
    mask_d = dow_oof == d
    if mask_d.sum() < 5:
        offsets[d] = 0.0; continue
    r = resid_oof[mask_d]
    offsets[d] = np.clip(np.median(r), -15, 15)
    print(f"  DOW {d}: offset={offsets[d]:.3f}")

cal1_count = blend_count.copy()
for d, off in offsets.items():
    cal1_count[dow_oof == d] += off
cost_cal1 = asymmetric_cost(y_blend, cal1_count)
print(f"\nAfter Stage 1 AsymCost: {cost_cal1:.4f}")

# Stage 2: shrinkage toward per-DOW mean (reduce variance)
dow_mean_actual = {}
for d in [6, 0, 1, 2, 3]:
    mask_d = dow_oof == d
    if mask_d.sum() > 0:
        dow_mean_actual[d] = np.mean(y_blend[mask_d])

def cal_cost_shrink(params):
    lam = params[0]
    pred = cal1_count.copy()
    for i, d in enumerate([6, 0, 1, 2, 3]):
        mask_d = dow_oof == d
        if mask_d.sum() > 0:
            pred[mask_d] = lam * pred[mask_d] + (1 - lam) * dow_mean_actual[d]
    return asymmetric_cost(y_blend, pred)

res = minimize(cal_cost_shrink, [0.8], method='L-BFGS-B', bounds=[(0.3, 1.0)])
lam_opt = res.x[0]
print(f"\nOptimal shrinkage lambda: {lam_opt:.4f}")

cal2_count = cal1_count.copy()
for d in [6, 0, 1, 2, 3]:
    mask_d = dow_oof == d
    if mask_d.sum() > 0:
        cal2_count[mask_d] = lam_opt * cal1_count[mask_d] + (1 - lam_opt) * dow_mean_actual[d]

clip_lo = np.percentile(feat['employees_count'].values, 3)
clip_hi = feat['employees_count'].max() * 1.15
cal2_count = np.clip(cal2_count, clip_lo, clip_hi)
cost_cal2 = asymmetric_cost(y_blend, cal2_count)
print(f"After Stage 2 AsymCost: {cost_cal2:.4f}")

  DOW 6: offset=-5.721
  DOW 0: offset=-7.519
  DOW 1: offset=-9.740
  DOW 2: offset=-5.118
  DOW 3: offset=-8.860

After Stage 1 AsymCost: 26.2073

Optimal shrinkage lambda: 0.7869
After Stage 2 AsymCost: 24.5750


## Full Retrain + Test Predictions


In [12]:
TEST_SIZE = max(40, n // 8)
train_df = feat.iloc[:n - TEST_SIZE].copy()
test_df  = feat.iloc[n - TEST_SIZE:].copy()
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
print(f"Test: {test_df.Date.min().date()} to {test_df.Date.max().date()}")

ok_tr = ~(np.isnan(train_df[TARGET].values) | np.isnan(train_df[FEAT_COLS].values).any(axis=1))
X_full = train_df.loc[ok_tr, FEAT_COLS].values
y_full = train_df.loc[ok_tr, TARGET].values
w_full = train_df.loc[ok_tr, 'office_present_pred'].values

ok_te = ~(np.isnan(test_df[TARGET].values) | np.isnan(test_df[FEAT_COLS].values).any(axis=1))
X_test = test_df[FEAT_COLS].values
y_test_actual = test_df['employees_count'].values
w_test = test_df['office_present_pred'].values

avg_params = {}
for k in all_best_params[0].keys():
    vals = [bp[k] for bp in all_best_params if k in bp]
    avg_params[k] = np.mean(vals)

# --- Collect trained model objects for deployment ---
lgb_trained_models = []   # list of (alpha, seed, lgb.Booster)
xgb_trained_models = []   # list of (alpha, seed, xgb.XGBRegressor)
cb_trained_models = []    # list of (seed, CatBoostRegressor)


# --- Collect trained model objects for deployment ---
lgb_trained_models = []   # list of (alpha, seed, lgb.Booster)
xgb_trained_models = []   # list of (alpha, seed, xgb.XGBRegressor)
cb_trained_models = []    # list of (seed, CatBoostRegressor)

# LGB final
lgb_final = []
for alpha in LGB_ALPHAS:
    seed_preds = []
    for seed in LGB_SEEDS:
        p = dict(objective='quantile', alpha=alpha,
                 learning_rate=avg_params['lr'], num_leaves=int(avg_params['nl']),
                 max_depth=int(avg_params['md']), min_child_samples=int(avg_params['mcs']),
                 subsample=avg_params['ss'], colsample_bytree=avg_params['cs'],
                 reg_alpha=avg_params['ra'], reg_lambda=avg_params['rl'],
                 verbose=-1, seed=seed, n_jobs=-1)
        ds = lgb.Dataset(X_full, label=y_full, weight=w_full)
        m = lgb.train(p, ds, num_boost_round=500)
        pred = np.full(len(test_df), np.nan)
        pred[ok_te] = m.predict(X_test[ok_te])
        seed_preds.append(pred)
        lgb_trained_models.append((alpha, seed, m))
    lgb_final.append(np.nanmean(seed_preds, axis=0))
lgb_final_ratio = np.array(lgb_final).T @ lgb_weights_arr

# XGB final
xgb_final = []
for alpha in XGB_ALPHAS:
    seed_preds = []
    for seed in XGB_SEEDS:
        p = dict(objective='reg:quantileerror', quantile_alpha=alpha,
                 learning_rate=best_xgb['lr'], max_depth=best_xgb['md'],
                 subsample=best_xgb['ss'], colsample_bytree=best_xgb['cs'],
                 reg_alpha=best_xgb['ra'], reg_lambda=best_xgb['rl'],
                 verbosity=0, seed=seed, n_jobs=-1)
        m = xgb.XGBRegressor(**p, n_estimators=500)
        m.fit(X_full, y_full, sample_weight=w_full, verbose=False)
        pred = np.full(len(test_df), np.nan)
        pred[ok_te] = m.predict(X_test[ok_te])
        seed_preds.append(pred)
        xgb_trained_models.append((alpha, seed, m))
    xgb_final.append(np.nanmean(seed_preds, axis=0))
xgb_final_ratio = np.mean(xgb_final, axis=0)

test_ratio = best_blend_w[0] * lgb_final_ratio + best_blend_w[1] * xgb_final_ratio
if HAS_CATBOOST and best_blend_w.shape[0] > 2:
    cb_final = []
    for seed in CB_SEEDS:
        m = CatBoostRegressor(loss_function="Quantile:alpha=0.71",
                              learning_rate=best_cb.get('lr', 0.05),
                              depth=best_cb.get('d', 6),
                              l2_leaf_reg=best_cb.get('l2', 3.0),
                              iterations=500, verbose=0, random_seed=seed)
        m.fit(X_full, y_full, sample_weight=w_full, verbose=0)
        pred = np.full(len(test_df), np.nan)
        pred[ok_te] = m.predict(X_test[ok_te])
        cb_final.append(pred)
        cb_trained_models.append((seed, m))
    test_ratio += best_blend_w[2] * np.nanmean(cb_final, axis=0)

# Calibration on test (in count space)
test_dow = test_df['dow'].values
cal_test = test_ratio * w_test
for d, off in offsets.items():
    cal_test[test_dow == d] += off

test_dow_mean = {}
for d in [6, 0, 1, 2, 3]:
    mask = test_dow == d
    if mask.sum() > 0:
        test_dow_mean[d] = np.mean(cal_test[mask])
for d in [6, 0, 1, 2, 3]:
    mask = test_dow == d
    if mask.sum() > 0 and d in dow_mean_actual:
        cal_test[mask] = lam_opt * cal_test[mask] + (1 - lam_opt) * dow_mean_actual[d]

test_count = np.clip(cal_test, clip_lo, clip_hi)
test_count_int = np.round(test_count).astype(int)
cost_test = asymmetric_cost(y_test_actual, test_count)
rmse_test = np.sqrt(mean_squared_error(y_test_actual, test_count))
mae_test = mean_absolute_error(y_test_actual, test_count)

print(f"\nTest AsymCost: {cost_test:.4f}")
print(f"Test RMSE: {rmse_test:.2f}  |  MAE: {mae_test:.2f}")

Train: 529  |  Test: 75
Test: 2024-09-18 to 2024-12-31

Test AsymCost: 18.9317
Test RMSE: 24.32  |  MAE: 17.04


## Diagnostics


In [13]:
pred_std = np.std(cal2_count)
train_std = np.std(y_blend)
var_ratio = pred_std / (train_std + 1e-9)
print(f"Variance ratio: {var_ratio:.4f}  (target: 0.55-0.95)")

under_mask = cal2_count < y_blend
over_mask  = cal2_count > y_blend
under_cost = np.sum(2 * (y_blend[under_mask] - cal2_count[under_mask]))
over_cost  = np.sum(cal2_count[over_mask] - y_blend[over_mask])
ratio_uo = under_cost / (over_cost + 1e-9)
print(f"Under-prediction cost: {under_cost:.1f}")
print(f"Over-prediction cost:  {over_cost:.1f}")
print(f"Under/Over ratio: {ratio_uo:.3f}  (target: 1.5-2.0)")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(y_blend, cal2_count, alpha=0.3, s=8)
lims = [min(y_blend.min(), cal2_count.min()), max(y_blend.max(), cal2_count.max())]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'OOF Pred vs Actual (AsymCost={cost_cal2:.4f})')

resid_all = y_blend - cal2_count
dow_order = [6, 0, 1, 2, 3]
dow_labels = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu']
dow_resid = [resid_all[dow_oof == d].mean() for d in dow_order]
colors = ['#2ecc71' if r >= 0 else '#e74c3c' for r in dow_resid]
axes[1].bar(dow_labels, dow_resid, color=colors, alpha=0.7)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Mean Residual by DOW')

axes[2].hist(y_blend, bins=30, alpha=0.5, label='Train actual', density=True)
axes[2].hist(cal2_count, bins=30, alpha=0.5, label='OOF predicted', density=True)
axes[2].set_title('Distribution Comparison')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'main_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/main_diagnostics.png")

Variance ratio: 0.7985  (target: 0.55-0.95)
Under-prediction cost: 2482.7
Over-prediction cost:  1203.6
Under/Over ratio: 2.063  (target: 1.5-2.0)
Saved: reports/figures/main_diagnostics.png


## Save Outputs


In [14]:
final_df = pd.DataFrame({
    'Date': test_df['Date'].values,
    'cantine_presence_pred': test_count_int,
    'office_presence_pred': np.round(w_test).astype(int),
    'ratio_pred': test_ratio
})
final_df.to_csv(PROC_DIR / 'predictions_final.csv', index=False)
print(f"Saved: data/processed/predictions_final.csv ({len(final_df)} rows)")

# --- Save metadata pkl files (backward-compatible) ---
with open(MODEL_DIR / 'lgb_models.pkl', 'wb') as f:
    pickle.dump({'alphas': LGB_ALPHAS, 'seeds': LGB_SEEDS,
                 'alpha_weights': alpha_weights, 'params': all_best_params[-1],
                 'feat_cols': FEAT_COLS}, f)
print("Saved: models/lgb_models.pkl")

with open(MODEL_DIR / 'xgb_models.pkl', 'wb') as f:
    pickle.dump({'alphas': XGB_ALPHAS, 'seeds': XGB_SEEDS,
                 'params': best_xgb, 'feat_cols': FEAT_COLS}, f)
print("Saved: models/xgb_models.pkl")

if HAS_CATBOOST:
    with open(MODEL_DIR / 'catboost_models.pkl', 'wb') as f:
        pickle.dump({'seeds': CB_SEEDS, 'params': best_cb, 'feat_cols': FEAT_COLS}, f)
    print("Saved: models/catboost_models.pkl")

# --- Save deployment bundle (everything needed for API inference) ---
deployment = {
    'lgb_models': lgb_trained_models,       # list of (alpha, seed, Booster)
    'xgb_models': xgb_trained_models,       # list of (alpha, seed, XGBRegressor)
    'cb_models': cb_trained_models,          # list of (seed, CatBoostRegressor)
    'lgb_alpha_weights': alpha_weights,     # dict {'a0.65': w, ...}
    'lgb_weights_arr': lgb_weights_arr,     # np.array shape (6,)
    'best_blend_w': best_blend_w,           # np.array [w_lgb, w_xgb, w_cb]
    'offsets': offsets,                     # dict {dow: offset}
    'dow_mean_actual': dow_mean_actual,     # dict {dow: mean_count}
    'lam_opt': lam_opt,                     # float
    'clip_lo': float(clip_lo),              # float
    'clip_hi': float(clip_hi),              # float
    'feat_cols': FEAT_COLS,                 # list of 126 feature names
    'lgb_alphas': LGB_ALPHAS,              # list of 6 floats
    'lgb_seeds': LGB_SEEDS,                # list of 4 ints
    'xgb_alphas': XGB_ALPHAS,              # list of 3 floats
    'xgb_seeds': XGB_SEEDS,                # list of 3 ints
    'cb_seeds': CB_SEEDS,                  # list of 3 ints
    'avg_lgb_params': avg_params,           # dict of averaged LGB params
    'xgb_params': best_xgb,                # dict of XGB params
    'cb_params': best_cb,                   # dict of CatBoost params
    'has_catboost': HAS_CATBOOST,
    'target': TARGET,                       # 'ratio_capped'
    'oof_metrics': {
        'AsymCost': float(cost_cal2),
        'RMSE': float(np.sqrt(mean_squared_error(y_blend, cal2_count))),
        'MAE': float(mean_absolute_error(y_blend, cal2_count)),
    },
}
bundle_path = MODEL_DIR / '_deployment.pkl'
with open(bundle_path, 'wb') as f:
    pickle.dump(deployment, f)
print(f"Saved: models/_deployment.pkl ({len(lgb_trained_models)} LGB + {len(xgb_trained_models)} XGB + {len(cb_trained_models)} CB models)")
print(f"  Blend weights: LGB={best_blend_w[0]:.4f}, XGB={best_blend_w[1]:.4f}, CB={best_blend_w[2] if len(best_blend_w) > 2 else 0:.4f}")
print(f"  Calibration: lambda={lam_opt:.4f}, clip=[{clip_lo:.1f}, {clip_hi:.1f}]")
print(f"  DOW offsets: {offsets}")

Saved: data/processed/predictions_final.csv (75 rows)
Saved: models/lgb_models.pkl
Saved: models/xgb_models.pkl
Saved: models/catboost_models.pkl
Saved: models/_deployment.pkl (24 LGB + 9 XGB + 3 CB models)
  Blend weights: LGB=0.0159, XGB=0.1905, CB=0.7937
  Calibration: lambda=0.7869, clip=[243.1, 614.1]
  DOW offsets: {6: np.float64(-5.720513261938748), 0: np.float64(-7.519213546290871), 1: np.float64(-9.740116451730103), 2: np.float64(-5.117636396627063), 3: np.float64(-8.85978807287728)}


## Summary


In [15]:
print("=" * 60)
print("MAIN MODEL SUMMARY")
print("=" * 60)
print(f"Architecture:      {ARCH_NAME}")
print(f"OOF AsymCost:      {cost_cal2:.4f}")
print(f"Test AsymCost:     {cost_test:.4f}")
print(f"Test RMSE:         {rmse_test:.2f}")
print(f"Test MAE:          {mae_test:.2f}")
print(f"Variance ratio:    {var_ratio:.4f}  (target: 0.55-0.95)")
print(f"Under/Over ratio:  {ratio_uo:.3f}  (target: 1.5-2.0)")
print()
print("Blend weights:")
for name, wi in zip(model_names, best_blend_w):
    print(f"  {name}: {wi:.4f}")
print()
print("Calibration:")
print(f"  Shrinkage lambda: {lam_opt:.4f}")
dow_names_cal = {6:'Sun', 0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu'}
for d, off in offsets.items():
    print(f"  DOW {dow_names_cal[d]} offset: {off:.3f}")
print()
print("Alpha weights:")
for k, v in alpha_weights.items():
    print(f"  {k}: {v:.4f}")
print("=" * 60)

MAIN MODEL SUMMARY
Architecture:      B (Cascade)
OOF AsymCost:      24.5750
Test AsymCost:     18.9317
Test RMSE:         24.32
Test MAE:          17.04
Variance ratio:    0.7985  (target: 0.55-0.95)
Under/Over ratio:  2.063  (target: 1.5-2.0)

Blend weights:
  LGB: 0.0159
  XGB: 0.1905
  CatBoost: 0.7937

Calibration:
  Shrinkage lambda: 0.7869
  DOW Sun offset: -5.721
  DOW Mon offset: -7.519
  DOW Tue offset: -9.740
  DOW Wed offset: -5.118
  DOW Thu offset: -8.860

Alpha weights:
  a0.65: 0.1240
  a0.68: 0.1283
  a0.71: 0.1353
  a0.74: 0.0000
  a0.77: 0.5316
  a0.80: 0.0808
